# Reconciliation: ledger.csv vs gateway.csv

This notebook performs a reconciliation between `01_data/raw/ledger.csv` and `01_data/raw/gateway.csv` and writes reconciliation outputs to `01_data/processed/` and a summary JSON to `04_python/summary_metrics.json`.

Steps:
- Load both files
- Check duplicates and nulls
- Identify records missing in gateway
- Identify records missing in ledger
- Identify amount mismatches
- Identify status mismatches
- Build final reconciliation report
- Generate summary metrics

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime

# Locate project root by looking for the `01_data` folder in parents
cwd = Path.cwd()
project_root = None
for p in [cwd] + list(cwd.parents):
    if (p / '01_data').exists():
        project_root = p
        break
if project_root is None:
    project_root = cwd

raw_dir = project_root / '01_data' / 'raw'
processed_dir = project_root / '01_data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

ledger_fp = raw_dir / 'ledger.csv'
gateway_fp = raw_dir / 'gateway.csv'
summary_json_fp = project_root / '04_python' / 'summary_metrics.json'

print('Project root:', project_root)
print('Ledger file:', ledger_fp)
print('Gateway file:', gateway_fp)
print('Processed dir:', processed_dir)

Project root: d:\Python\BitSom\quickpay_data_analysis
Ledger file: d:\Python\BitSom\quickpay_data_analysis\01_data\raw\ledger.csv
Gateway file: d:\Python\BitSom\quickpay_data_analysis\01_data\raw\gateway.csv
Processed dir: d:\Python\BitSom\quickpay_data_analysis\01_data\processed


In [3]:
# Load CSVs
ledger = pd.read_csv(ledger_fp, parse_dates=['transaction_date'])
gateway = pd.read_csv(gateway_fp, parse_dates=['transaction_date'])

# Normalize basic types
for df in (ledger, gateway):
    df['transaction_id'] = df['transaction_id'].astype(str)
    df['merchant_id'] = df['merchant_id'].astype(str)
    df['amount_usd'] = pd.to_numeric(df['amount_usd'], errors='coerce')
    df['status'] = df['status'].astype(str)

print('Ledger rows:', len(ledger))
print('Gateway rows:', len(gateway))
print('Ledger head:')
print(ledger.head(5))
print('Gateway head:')
print(gateway.head(5))

Ledger rows: 10
Gateway rows: 9
Ledger head:
  transaction_id transaction_date merchant_id  amount_usd   status  \
0           R001       2026-03-01        M001      1200.0  success   
1           R002       2026-03-01        M002       850.0  success   
2           R003       2026-03-02        M001       500.0  success   
3           R004       2026-03-02        M003      2100.0  success   
4           R005       2026-03-03        M004      7200.0  success   

  payment_method  
0            UPI  
1           Card  
2         Wallet  
3           Card  
4           Card  
Gateway head:
  transaction_id transaction_date merchant_id  amount_usd   status  \
0           R001       2026-03-01        M001      1200.0  success   
1           R002       2026-03-01        M002       900.0  success   
2           R003       2026-03-02        M001       500.0  success   
3           R005       2026-03-03        M004      7200.0   failed   
4           R006       2026-03-03        M002       950.

In [4]:
# Check duplicates and nulls
dupes_ledger = ledger[ledger.duplicated(subset=['transaction_id'], keep=False)]
dupes_gateway = gateway[gateway.duplicated(subset=['transaction_id'], keep=False)]

print('Ledger duplicate count:', len(dupes_ledger))
print('Gateway duplicate count:', len(dupes_gateway))

print('Ledger nulls:', ledger.isnull().sum())
print('Gateway nulls:', gateway.isnull().sum())

Ledger duplicate count: 0
Gateway duplicate count: 0
Ledger nulls: transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64
Gateway nulls: transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64


In [5]:
# Merge to detect presence and mismatches
merged = ledger.merge(gateway, on='transaction_id', how='outer', suffixes=('_ledger', '_gateway'), indicator=True)

# Missing in gateway -> present in ledger but not in gateway
missing_in_gateway = merged[merged['_merge'] == 'left_only'].copy()
missing_in_gateway = missing_in_gateway[[
    'transaction_id',
    'transaction_date_ledger',
    'merchant_id_ledger',
    'amount_usd_ledger',
    'status_ledger',
    'payment_method_ledger'
]]
missing_in_gateway.columns = ['transaction_id','transaction_date','merchant_id','amount_usd','status','payment_method']
missing_in_gateway.to_csv(processed_dir / 'missing_in_gateway.csv', index=False)
print('Missing in gateway count:', len(missing_in_gateway))

# Missing in ledger -> present in gateway but not in ledger
missing_in_ledger = merged[merged['_merge'] == 'right_only'].copy()
missing_in_ledger = missing_in_ledger[[
    'transaction_id',
    'transaction_date_gateway',
    'merchant_id_gateway',
    'amount_usd_gateway',
    'status_gateway',
    'payment_method_gateway'
]]
missing_in_ledger.columns = ['transaction_id','transaction_date','merchant_id','amount_usd','status','payment_method']
missing_in_ledger.to_csv(processed_dir / 'missing_in_ledger.csv', index=False)
print('Missing in ledger count:', len(missing_in_ledger))

Missing in gateway count: 2
Missing in ledger count: 1


In [6]:
# Amount mismatches for transactions present in both
both = merged[merged['_merge'] == 'both'].copy()

def _is_amount_mismatch(a, b):
    if pd.isna(a) and pd.isna(b):
        return False
    if pd.isna(a) or pd.isna(b):
        return True
    return not np.isclose(a, b)

both['amount_mismatch'] = both.apply(lambda r: _is_amount_mismatch(r['amount_usd_ledger'], r['amount_usd_gateway']), axis=1)
amount_mismatches = both[both['amount_mismatch']].copy()
# Normalize columns and always write a CSV (may be empty)
amount_mismatches_out = amount_mismatches[[
    'transaction_id',
    'transaction_date_ledger',
    'merchant_id_ledger',
    'amount_usd_ledger',
    'amount_usd_gateway',
    'status_ledger',
    'status_gateway'
]].copy()
amount_mismatches_out.columns = ['transaction_id','transaction_date','merchant_id','amount_usd_ledger','amount_usd_gateway','status_ledger','status_gateway']
if not amount_mismatches_out.empty:
    amount_mismatches_out['amount_diff'] = amount_mismatches_out['amount_usd_ledger'] - amount_mismatches_out['amount_usd_gateway']
amount_mismatches_out.to_csv(processed_dir / 'amount_mismatches.csv', index=False)
print('Amount mismatches count:', len(amount_mismatches_out))

Amount mismatches count: 2


In [7]:
# Status mismatches for transactions present in both
both['status_mismatch'] = both['status_ledger'].fillna('').str.lower() != both['status_gateway'].fillna('').str.lower()
status_mismatches = both[both['status_mismatch']].copy()
status_mismatches_out = status_mismatches[[
    'transaction_id',
    'transaction_date_ledger',
    'merchant_id_ledger',
    'status_ledger',
    'status_gateway'
]].copy()
status_mismatches_out.columns = ['transaction_id','transaction_date','merchant_id','status_ledger','status_gateway']
status_mismatches_out.to_csv(processed_dir / 'status_mismatches.csv', index=False)
print('Status mismatches count:', len(status_mismatches_out))

Status mismatches count: 1


In [8]:
# Build final reconciliation report
report = merged.copy()
report['transaction_date'] = report['transaction_date_ledger'].combine_first(report['transaction_date_gateway'])
report['merchant_id'] = report['merchant_id_ledger'].combine_first(report['merchant_id_gateway'])
report['amount_usd_ledger'] = report['amount_usd_ledger']
report['amount_usd_gateway'] = report['amount_usd_gateway']
report['status_ledger'] = report['status_ledger']
report['status_gateway'] = report['status_gateway']

def _amount_match(a, b):
    if pd.isna(a) and pd.isna(b):
        return True
    if pd.isna(a) or pd.isna(b):
        return False
    return np.isclose(a, b)

report['amount_match'] = report.apply(lambda r: _amount_match(r['amount_usd_ledger'], r['amount_usd_gateway']), axis=1)
report['status_match'] = report['status_ledger'].fillna('').str.lower() == report['status_gateway'].fillna('').str.lower()
report['presence'] = report['_merge'].map({'both':'both','left_only':'missing_in_gateway','right_only':'missing_in_ledger'})

def _recon_status(r):
    if r['presence'] != 'both':
        return r['presence']
    if not r['amount_match']:
        return 'amount_mismatch'
    if not r['status_match']:
        return 'status_mismatch'
    return 'matched'

report['recon_status'] = report.apply(_recon_status, axis=1)
report_out = report[[
    'transaction_id', 'transaction_date', 'merchant_id',
    'amount_usd_ledger', 'amount_usd_gateway', 'amount_match',
    'status_ledger', 'status_gateway', 'status_match',
    'presence', 'recon_status'
]].copy()
report_out.to_csv(processed_dir / 'reconciliation_report.csv', index=False)
print('Wrote reconciliation_report.csv with', len(report_out), 'rows')

Wrote reconciliation_report.csv with 11 rows


In [9]:
# Summary metrics and write JSON
metrics = {
    'run_timestamp': datetime.utcnow().isoformat() + 'Z',
    'ledger_count': int(ledger['transaction_id'].nunique()),
    'gateway_count': int(gateway['transaction_id'].nunique()),
    'missing_in_gateway_count': int(missing_in_gateway['transaction_id'].nunique()),
    'missing_in_ledger_count': int(missing_in_ledger['transaction_id'].nunique()),
    'amount_mismatches_count': int(amount_mismatches_out.shape[0]),
    'status_mismatches_count': int(status_mismatches_out.shape[0]),
    'matched_count': int(report_out[report_out['recon_status'] == 'matched'].shape[0])
}
summary_json_fp.parent.mkdir(parents=True, exist_ok=True)
with open(summary_json_fp, 'w') as fh:
    json.dump(metrics, fh, indent=2)

print('Summary metrics:')
print(json.dumps(metrics, indent=2))

Summary metrics:
{
  "run_timestamp": "2026-05-04T14:40:25.835847Z",
  "ledger_count": 10,
  "gateway_count": 9,
  "missing_in_gateway_count": 2,
  "missing_in_ledger_count": 1,
  "amount_mismatches_count": 2,
  "status_mismatches_count": 1,
  "matched_count": 5
}


C:\Users\S.Shashank Kumar\AppData\Local\Temp\ipykernel_23224\508950876.py:3: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'run_timestamp': datetime.utcnow().isoformat() + 'Z',


In [ ]:
# Dashboard / BI outputs derived from reconciliation
cleaned = report.copy()
cleaned['transaction_date'] = pd.to_datetime(cleaned['transaction_date'], utc=True, errors='coerce')
cleaned['merchant_id'] = cleaned['merchant_id'].astype(str)

# Payment method - combine ledger/gateway fields if present
if 'payment_method_ledger' in cleaned.columns and 'payment_method_gateway' in cleaned.columns:
    cleaned['payment_method'] = cleaned['payment_method_ledger'].combine_first(cleaned['payment_method_gateway'])
elif 'payment_method_ledger' in cleaned.columns:
    cleaned['payment_method'] = cleaned['payment_method_ledger']
elif 'payment_method_gateway' in cleaned.columns:
    cleaned['payment_method'] = cleaned['payment_method_gateway']
else:
    cleaned['payment_method'] = None

# Amount diff
if 'amount_usd_ledger' in cleaned.columns and 'amount_usd_gateway' in cleaned.columns:
    cleaned['amount_diff'] = cleaned['amount_usd_ledger'] - cleaned['amount_usd_gateway']
else:
    cleaned['amount_diff'] = None

# Normalize some boolean flags
if 'amount_match' not in cleaned.columns:
    cleaned['amount_match'] = cleaned['amount_usd_ledger'].combine_first(cleaned['amount_usd_gateway']).notna()
if 'status_match' not in cleaned.columns:
    cleaned['status_match'] = cleaned['status_ledger'].fillna('').str.lower() == cleaned['status_gateway'].fillna('').str.lower()

# Select columns for cleaned_transactions (keep available ones)
keep_cols = ['transaction_id','transaction_date','merchant_id','amount_usd_ledger','amount_usd_gateway','amount_diff','amount_match','status_ledger','status_gateway','status_match','payment_method','presence','recon_status']
keep_cols = [c for c in keep_cols if c in cleaned.columns]
cleaned_transactions = cleaned[keep_cols].copy()
cleaned_transactions.to_csv(processed_dir / 'cleaned_transactions.csv', index=False)
print('Wrote cleaned_transactions.csv with', len(cleaned_transactions), 'rows')

# Load merchant master (if available) to enrich merchant metadata
merchant_master_fp = raw_dir / 'merchant_master.csv'
merchant_master = None
if merchant_master_fp.exists():
    merchant_master = pd.read_csv(merchant_master_fp, dtype={'merchant_id':str})
    if 'merchant_id' in merchant_master.columns:
        merchant_master['merchant_id'] = merchant_master['merchant_id'].astype(str)
    if 'default_region' in merchant_master.columns and 'region' not in merchant_master.columns:
        merchant_master = merchant_master.rename(columns={'default_region': 'region'})
    if 'merchant_name' in merchant_master.columns and 'region' in merchant_master.columns:
        cleaned_transactions = cleaned_transactions.merge(
            merchant_master[['merchant_id', 'merchant_name', 'region']],
            on='merchant_id',
            how='left'
        )

# Daily summary (one row per UTC date)
df = cleaned_transactions.copy()
df['date'] = pd.to_datetime(df['transaction_date'], utc=True, errors='coerce').dt.date
daily_rows = []
for d in sorted(df['date'].dropna().unique()):
    sub = df[df['date'] == d]
    row = {
        'date': d.isoformat(),
        'transactions_count': int(sub['transaction_id'].nunique()),
        'total_amount_ledger': float(sub['amount_usd_ledger'].sum(skipna=True)) if 'amount_usd_ledger' in sub.columns else None,
        'total_amount_gateway': float(sub['amount_usd_gateway'].sum(skipna=True)) if 'amount_usd_gateway' in sub.columns else None,
        'matched_count': int((sub['recon_status'] == 'matched').sum()) if 'recon_status' in sub.columns else 0,
        'amount_mismatches_count': int((sub['amount_match'] == False).sum()) if 'amount_match' in sub.columns else 0,
        'status_mismatches_count': int((sub['status_match'] == False).sum()) if 'status_match' in sub.columns else 0,
        'missing_in_gateway_count': int((sub['presence'] == 'missing_in_gateway').sum()) if 'presence' in sub.columns else 0,
        'missing_in_ledger_count': int((sub['presence'] == 'missing_in_ledger').sum()) if 'presence' in sub.columns else 0,
    }
    daily_rows.append(row)
daily_summary = pd.DataFrame.from_records(daily_rows)
daily_summary.to_csv(processed_dir / 'daily_summary.csv', index=False)
print('Wrote daily_summary.csv with', len(daily_summary), 'rows')

# Payment method breakdown
if 'payment_method' in cleaned_transactions.columns:
    pm = cleaned_transactions.groupby('payment_method').agg(
        transactions_count=('transaction_id','nunique'),
        total_amount_ledger=('amount_usd_ledger','sum'),
        total_amount_gateway=('amount_usd_gateway','sum')
    ).reset_index()
    pm['transactions_share'] = pm['transactions_count'] / pm['transactions_count'].sum() if pm['transactions_count'].sum() else 0
    pm.to_csv(processed_dir / 'payment_method_breakdown.csv', index=False)
    print('Wrote payment_method_breakdown.csv with', len(pm), 'rows')
else:
    print('No payment_method column; skipped payment_method_breakdown')

# Region breakdown (requires merchant_master with region)
region_src = None
if 'region' in cleaned_transactions.columns:
    region_src = cleaned_transactions
elif merchant_master is not None and 'region' in merchant_master.columns:
    region_src = cleaned_transactions.merge(merchant_master[['merchant_id','region']], on='merchant_id', how='left')
if region_src is not None:
    region = region_src.groupby('region').agg(
        transactions_count=('transaction_id','nunique'),
        total_amount_ledger=('amount_usd_ledger','sum'),
        total_amount_gateway=('amount_usd_gateway','sum')
    ).reset_index().rename(columns={'region':'merchant_region'})
    region['transactions_share'] = region['transactions_count'] / region['transactions_count'].sum() if region['transactions_count'].sum() else 0
    region.to_csv(processed_dir / 'region_breakdown.csv', index=False)
    print('Wrote region_breakdown.csv with', len(region), 'rows')
else:
    print('merchant_master missing or has no region column; skipped region_breakdown')

# Merchant performance summary
mp = cleaned_transactions.copy()
risk_fp = processed_dir / 'merchant_risk_summary.csv'
use_risk_fp = False
if risk_fp.exists():
    tmp = pd.read_csv(risk_fp, dtype={'merchant_id':str})
    # detect if this looks like an aggregated per-merchant risk file
    if 'merchant_id' in tmp.columns and 'transactions_count' in tmp.columns:
        grp = tmp
        use_risk_fp = True
if use_risk_fp:
    # Ensure numeric columns are present and compute rates if missing
    if 'transactions_count' in grp.columns and 'matched_count' in grp.columns:
        grp['matched_rate'] = grp['matched_count'] / grp['transactions_count']
    if 'transactions_count' in grp.columns and 'amount_mismatches_count' in grp.columns:
        grp['amount_mismatch_rate'] = grp['amount_mismatches_count'] / grp['transactions_count']
    if 'transactions_count' in grp.columns and 'status_mismatches_count' in grp.columns:
        grp['status_mismatch_rate'] = grp['status_mismatches_count'] / grp['transactions_count']
    if merchant_master is not None and 'merchant_id' in merchant_master.columns and 'merchant_name' not in grp.columns:
        grp = grp.merge(merchant_master[['merchant_id','merchant_name']], on='merchant_id', how='left')
    grp = grp.sort_values('transactions_count', ascending=False) if 'transactions_count' in grp.columns else grp
    grp.to_csv(processed_dir / 'merchant_performance_summary.csv', index=False)
    print('Wrote merchant_performance_summary.csv with', len(grp), 'rows (from merchant_risk_summary.csv)')
else:
    if 'merchant_id' in mp.columns:
        grp = mp.groupby('merchant_id').agg(
            transactions_count=('transaction_id','nunique'),
            total_amount_ledger=('amount_usd_ledger','sum'),
            total_amount_gateway=('amount_usd_gateway','sum'),
            matched_count=('recon_status', lambda s: (s == 'matched').sum()) if 'recon_status' in mp.columns else ('transaction_id','nunique'),
            amount_mismatches_count=('amount_match', lambda s: (s == False).sum()) if 'amount_match' in mp.columns else ('transaction_id','nunique'),
            status_mismatches_count=('status_match', lambda s: (s == False).sum()) if 'status_match' in mp.columns else ('transaction_id','nunique')
        )
        grp = grp.reset_index()
        if merchant_master is not None and 'merchant_id' in merchant_master.columns:
            grp = grp.merge(merchant_master[['merchant_id','merchant_name']], on='merchant_id', how='left')
        # compute rates
        grp['matched_rate'] = grp['matched_count'] / grp['transactions_count']
        grp['amount_mismatch_rate'] = grp['amount_mismatches_count'] / grp['transactions_count']
        grp['status_mismatch_rate'] = grp['status_mismatches_count'] / grp['transactions_count']
        grp = grp.sort_values('transactions_count', ascending=False)
        grp.to_csv(processed_dir / 'merchant_performance_summary.csv', index=False)
        print('Wrote merchant_performance_summary.csv with', len(grp), 'rows')
    else:
        print('No merchant_id column; skipped merchant_performance_summary')

# Merchant risk summary (simple weighted score)
if 'merchant_id' in mp.columns:
    risk = grp.copy() if 'grp' in locals() else pd.DataFrame()
    if not risk.empty:
        # weights: amount mismatch x2, missing x2 (approx via amount/status mismatch proxy), status mismatch x1
        risk['missing_rate'] = (risk.get('missing_in_gateway_count', 0) + risk.get('missing_in_ledger_count', 0)) / risk['transactions_count'] if 'missing_in_gateway_count' in risk.columns or 'missing_in_ledger_count' in risk.columns else 0
        risk['risk_score'] = (risk['amount_mismatch_rate'].fillna(0) * 2) + (risk['status_mismatch_rate'].fillna(0) * 1) + (risk['missing_rate'].fillna(0) * 2)
        risk = risk.sort_values('risk_score', ascending=False)
        risk.to_csv(processed_dir / 'merchant_risk_summary.csv', index=False)
        print('Wrote merchant_risk_summary.csv with', len(risk), 'rows')
    else:
        print('Merchant performance summary was empty; skipped merchant_risk_summary')
else:
    print('No merchant_id column; skipped merchant_risk_summary')

Wrote cleaned_transactions.csv with 11 rows
Wrote daily_summary.csv with 5 rows
Wrote payment_method_breakdown.csv with 4 rows


KeyError: 'region'

**Outputs written**:
- `01_data/processed/missing_in_gateway.csv`
- `01_data/processed/missing_in_ledger.csv`
- `01_data/processed/amount_mismatches.csv`
- `01_data/processed/status_mismatches.csv`
- `01_data/processed/reconciliation_report.csv`
- `04_python/summary_metrics.json`


## Part 4: JSON Normalization

Read `01_data/raw/api_response_sample.json`, flatten nested records into rows (one settlement per row), clean column names, convert datetime fields, and save to `01_data/processed/api_normalized.csv`.

In [11]:
import json
from pathlib import Path
import pandas as pd

# Paths
raw_json_fp = raw_dir / 'api_response_sample.json'
out_fp = processed_dir / 'api_normalized.csv'

# Load JSON
with open(raw_json_fp, 'r', encoding='utf-8') as fh:
    api = json.load(fh)

# Build records: one row per settlement with batch and merchant metadata
records = []
for batch in api.get('batches', []):
    batch_id = batch.get('batch_id')
    merchant = batch.get('merchant', {}) or {}
    for s in batch.get('settlements', []):
        rec = {}
        rec['batch_id'] = batch_id
        # merchant fields
        rec['merchant_id'] = merchant.get('merchant_id')
        rec['merchant_name'] = merchant.get('merchant_name')
        rec['merchant_region'] = merchant.get('region')
        # settlement top-level fields
        rec['settlement_id'] = s.get('settlement_id')
        rec['amount_usd'] = s.get('amount_usd')
        rec['status'] = s.get('status')
        rec['processed_at'] = s.get('processed_at')
        # bank nested
        bank = s.get('bank') or {}
        rec['bank_name'] = bank.get('name')
        rec['bank_country'] = bank.get('country')
        records.append(rec)

# Add top-level generated_at and source to each row
generated_at = api.get('generated_at')
source = api.get('source')
for r in records:
    r['generated_at'] = generated_at
    r['source'] = source

df_api = pd.DataFrame.from_records(records)
print('Rows flattened:', len(df_api))
df_api.head()

Rows flattened: 6


,batch_id,merchant_id,merchant_name,merchant_region,settlement_id,amount_usd,status,processed_at,bank_name,bank_country,generated_at,source
0,B001,M001,Alpha Mart,APAC,S001,1520.5,settled,2026-03-07T08:10:00Z,Bank A,IN,2026-03-07T10:00:00Z,QuickPay Settlement API
1,B001,M001,Alpha Mart,APAC,S002,980.0,pending,2026-03-07T08:45:00Z,Bank A,IN,2026-03-07T10:00:00Z,QuickPay Settlement API
2,B001,M001,Alpha Mart,APAC,S003,640.0,settled,2026-03-07T09:15:00Z,Bank B,SG,2026-03-07T10:00:00Z,QuickPay Settlement API
3,B002,M004,Delta Travels,US,S004,2100.0,settled,2026-03-07T08:20:00Z,Bank C,US,2026-03-07T10:00:00Z,QuickPay Settlement API
4,B002,M004,Delta Travels,US,S005,500.0,failed,2026-03-07T08:50:00Z,Bank C,US,2026-03-07T10:00:00Z,QuickPay Settlement API


In [12]:
# Clean column names
df = df_api.copy()
df.columns = (df.columns.str.strip()
                .str.lower()
                .str.replace(' ', '_')
                .str.replace(r'[^0-9a-z_]', '', regex=True))

# Convert types
df['amount_usd'] = pd.to_numeric(df['amount_usd'], errors='coerce')
df['processed_at'] = pd.to_datetime(df['processed_at'], utc=True, errors='coerce')
df['generated_at'] = pd.to_datetime(df['generated_at'], utc=True, errors='coerce')

# Reorder columns to a sensible layout if present
cols_order = ['generated_at','source','batch_id','settlement_id','merchant_id','merchant_name','merchant_region','amount_usd','status','processed_at','bank_name','bank_country']
cols = [c for c in cols_order if c in df.columns] + [c for c in df.columns if c not in cols_order]
df = df[cols]

# Save normalized CSV
out_fp.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_fp, index=False)
print('Wrote', out_fp, 'with', len(df), 'rows')
df.head()

Wrote d:\Python\BitSom\quickpay_data_analysis\01_data\processed\api_normalized.csv with 6 rows


,generated_at,source,batch_id,settlement_id,merchant_id,merchant_name,merchant_region,amount_usd,status,processed_at,bank_name,bank_country
0,2026-03-07 10:00:00+00:00,QuickPay Settlement API,B001,S001,M001,Alpha Mart,APAC,1520.5,settled,2026-03-07 08:10:00+00:00,Bank A,IN
1,2026-03-07 10:00:00+00:00,QuickPay Settlement API,B001,S002,M001,Alpha Mart,APAC,980.0,pending,2026-03-07 08:45:00+00:00,Bank A,IN
2,2026-03-07 10:00:00+00:00,QuickPay Settlement API,B001,S003,M001,Alpha Mart,APAC,640.0,settled,2026-03-07 09:15:00+00:00,Bank B,SG
3,2026-03-07 10:00:00+00:00,QuickPay Settlement API,B002,S004,M004,Delta Travels,US,2100.0,settled,2026-03-07 08:20:00+00:00,Bank C,US
4,2026-03-07 10:00:00+00:00,QuickPay Settlement API,B002,S005,M004,Delta Travels,US,500.0,failed,2026-03-07 08:50:00+00:00,Bank C,US
